faire le benchmarking de ollama
j'ai utilisé la clé api fournit par kevin 

In [9]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OLLAMA_API_KEY")

url = "https://ollama.com/api/tags"

response = requests.get(
    url,
    headers={
        "Authorization": f"Bearer {api_key}"
    }
)

print("Status:", response.status_code)

data = response.json()

df = pd.DataFrame(data["models"])

display(df)

Status: 200


,name,model,modified_at,size,digest,details
0,glm-5.2,glm-5.2,2026-06-16T08:00:00-07:00,0,f553240f6dc4,"{'parent_model': '', 'format': '', 'family': '..."
1,minimax-m3,minimax-m3,2026-06-01T00:00:00Z,0,9ce5291d9c0e,"{'parent_model': '', 'format': '', 'family': '..."
2,glm-5.1,glm-5.1,2026-04-07T08:00:00-07:00,1507728316928,882e35812821,"{'parent_model': '', 'format': '', 'family': '..."
3,kimi-k3,kimi-k3,2026-07-27T08:00:00-07:00,1560860324864,d189309738f0,"{'parent_model': '', 'format': '', 'family': '..."
4,gpt-oss:120b,gpt-oss:120b,2025-08-05T00:00:00Z,65290180781,d98fe6ba01e6,"{'parent_model': '', 'format': '', 'family': '..."
5,qwen3.5:397b,qwen3.5:397b,2026-02-16T00:00:00Z,397000000000,b909ca2f1b7f,"{'parent_model': '', 'format': '', 'family': '..."
6,deepseek-v4-pro:0813,deepseek-v4-pro:0813,2026-08-13T08:00:00-07:00,892744322880,db976f1eacdd,"{'parent_model': '', 'format': '', 'family': '..."
7,nemotron-3-nano:30b,nemotron-3-nano:30b,2025-12-15T00:00:00Z,32645090390,a4b4f1851393,"{'parent_model': '', 'format': '', 'family': '..."
8,glm-5.3,glm-5.3,2026-08-28T08:00:00-07:00,755433728000,632dfda18c6d,"{'parent_model': '', 'format': '', 'family': '..."
9,deepseek-v4.1-flash,deepseek-v4.1-flash,2026-09-10T03:00:00Z,486723007816,97308b916fc3,"{'parent_model': '', 'format': '', 'family': '..."


maintenent on test quelque models

1-qwen2.5 

In [ ]:

# Benchmark Ollama
# Ce script mesure le temps de réponse et le débit pour un modèle donné.

import os
import time
import statistics
import requests
from dotenv import load_dotenv

load_dotenv()

MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
API_KEY = os.getenv("OLLAMA_API_KEY")
PROMPT = (
    "Explique en 5 phrases pourquoi les benchmarks sont utiles pour comparer "
    "des modèles de langage et comment mesurer la latence et le débit."
)
RUNS = 5

headers = {}
if OLLAMA_HOST.startswith("https://ollama.com"):
    if not API_KEY:
        raise ValueError("OLLAMA_API_KEY est requis pour l'API Ollama cloud")
    headers["Authorization"] = f"Bearer {API_KEY}"

# Vérifie que le service répond
try:
    r = requests.get(f"{OLLAMA_HOST}/api/tags", headers=headers, timeout=30)
    print("Status /api/tags:", r.status_code)
    print("Models disponibles:")
    print(r.json())
except Exception as e:
    print("Erreur de connexion à Ollama:", e)


def benchmark_once(prompt: str):
    start = time.perf_counter()
    response = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": MODEL, "prompt": prompt, "stream": False},
        headers=headers,
        timeout=300,
    )
    elapsed = time.perf_counter() - start

    try:
        body = response.json()
    except ValueError:
        body = {"response": response.text}

    answer = body.get("response", "")
    tokens = len(answer.split())
    tps = tokens / elapsed if elapsed > 0 else 0

    return {
        "status_code": response.status_code,
        "elapsed_sec": elapsed,
        "tokens": tokens,
        "tokens_per_sec": tps,
        "preview": answer[:120],
    }

results = [benchmark_once(PROMPT) for _ in range(RUNS)]

for i, r in enumerate(results, start=1):
    print(f"Run {i}: status={r['status_code']} | elapsed={r['elapsed_sec']:.2f}s | tokens={r['tokens']} | tokens/sec={r['tokens_per_sec']:.2f}")

latencies = [r["elapsed_sec"] for r in results]
tps = [r["tokens_per_sec"] for r in results]

print("\nMoyenne latence:", round(statistics.mean(latencies), 3), "s")
print("Médiane latence:", round(statistics.median(latencies), 3), "s")
print("P95 latence:", round(sorted(latencies)[min(len(latencies)-1, int(0.95 * len(latencies)) - 1)], 3), "s")
print("Moyenne tokens/sec:", round(statistics.mean(tps), 3))

# Exemple de sortie exploitable dans un DataFrame
import pandas as pd

df = pd.DataFrame(results)
print("\nDataFrame résultats:")
display(df)
